In [6]:
# 준비: Lab 6에서 가져온 모델·토크나이저 로드
import torch
import textwrap
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto"
)
model.eval()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [7]:
import torch

def chat(model,
         tokenizer,
         user_message,
         system_message="You are a helpful assistan",
         **generation_args):

    # 1. 역할별 메시지 구성
    # ssytem : 모델의 성격과 규칙, user : "사람의 질문", assistant : 이전 모델 답변
    # 왜 assistant 역할이 필요할까? -> LLM은 기억력이 없음 그래서 매 요청 마다 이전의 대화를 전부 다시 넣어줘야함!
    messages = [
        {"role" : "system", "content" :system_message},
        {"role": "user", "content": user_message},
    ]

    # 2. Qwen이 학습할 때 쓴 형식으로 변환하기
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False, # 토큰을 반환하는 것이 아니라, 사람이 읽을 수 있는 텍스트를 반환
        add_generation_prompt=True # assistant의 차례를 명시하는 부분
    )

    # 3. 문자열을 토큰ID 텐서로 바꾸고, gpu로 이동 ( 모델이 있는 곳 )
    inputs = tokenizer(
        text,
    return_tensors="pt").to(model.device)

    # 4. 추론이기 때문에 gradient 계산 삭제하기 ( 메모리 절약 )
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            **generation_args
        )

    # 5. generate는 프롬프트 + 답변을 같이 돌려줌 -> 프롬프트 길이 만큼 잘라내서 답변만 출력함
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]

    # 6. 토큰 ID -> 문자열, <|im_end|> 같은 특수 토큰을 제거
    return tokenizer.decode(
        new_tokens,
        skip_special_tokens = True
    )

In [8]:
# 디코딩 옵션 비교

# Greedy : 매번 확률 1등 토큰만 선택하도록 -> 항상 같은 답이 선택이 됨!
response = chat(model,
                tokenizer,
                "Recommend a Thai restaurant meal.",
                max_new_tokens=80,
                do_sample=False)
print("[Greedy]")
print(textwrap.fill(response, width=80))

# Sampling : 매번 다른 답을 낼 수 있음, temperature(분포를 날카롭게/평평하게), top_k(상위 k개만 후보), top_p(누적확률 p까지만 후보)로 다양성을 조절 가능
response = chat(model,
                tokenizer,
                "Suggest a dinner menu.",
                max_new_tokens=80,
                do_sample=True,
                temperature=0.8,
                top_k=20)
print("\n[Sampling]")
print(textwrap.fill(response, width=80))

[Greedy]
Here's a recommendation for a Thai restaurant meal:  Pad Thai with Tom Yum Soup
This dish is a classic combination of Thai cuisine that you won't want to miss.
The Pad Thai, which translates to "sticky rice noodles," features stir-fried
rice noodles in a flavorful sauce made from tamarind paste, fish sauce, and lime
juice. It's typically served with shrimp or tofu,

[Sampling]
Here's a suggested dinner menu for 4 people:  Appetizers: - Caprese Salad: Fresh
mozzarella, tomatoes, basil, olive oil and balsamic vinegar. - Shrimp Cocktail:
Shrimp sautéed with lemon juice and cocktail sauce.  Main Courses: - Grilled
Chicken Breast: Marinated in garlic, rosemary, and lemon juice and grilled to
perfection. -


In [10]:
# Qwen 챗봇 만들기

import ipywidgets as widgets
from IPython.display import display
import textwrap

history = []   # (user-사용자의 질문, assistant-모델의 답변) 튜플을 쌓아두는 대화 기록

user_input = widgets.Text(placeholder="Type your message here...",
                          description="You:",
                          layout=widgets.Layout(width="80%")
            )
send_button = widgets.Button(description="Send", button_style="primary")
chat_output = widgets.Output()

def send_message(_):
    message = user_input.value.strip() # 메시지의 공백을 제거

    if not message:
        return

    # 입력창 비우기
    user_input.value = ""

    # 프롬프트 생성하기, 최근 5번의 대화 기록과 이번 질문을 더하기
    prompt = ""
    for user, assistant in history[-5:]: # 질문, 답변 -> 튜플을 꺼내기
        prompt += f"User: {user}\nAssistant: {assistant}\n"
    prompt += f"User: {message}\nAssistant:" # 마직막은 Assistance: 로 끝내서 모델이 이어서 답하기 유도하기

    with chat_output:
        print(f"\nYou: {message}")

    # 위에서 생성한 프롬프트를 유저 메시지에 넣기
    response = chat(model, tokenizer, prompt, max_new_tokens=80,
                    do_sample=True, temperature=0.8, top_k=20, top_p=0.9)

    # 나눈 대화를 기록에 추가, 다음 질문에서 사용
    history.append((message, response))

    with chat_output:
        print("Assistant:")
        print(textwrap.fill(response, width=80))

send_button.on_click(send_message)   # 버튼 클릭 이벤트 연결
display(widgets.VBox([chat_output, widgets.HBox([user_input, send_button])]))